# E-commerce Sales Dataset

# Columns:
- Order ID, Product, Quantity Ordered, Price Each, Order Date, Purchase Address

This notebook generates synthetic monthly e-commerce sales data for 2019. The data has seasonal order volume, weighted product demand, purchase time peaks, weighted city addresses, and simple product bundle behavior.

This script is a notebook made from: https://github.com/Aniket-Mishra/Sales-Analysis-and-Reporting/blob/main/Data%20Generator/create_data.py, which I wrote some 6 years ago following a tutorial and updating the code by https://github.com/KeithGalli.

The code is mostly the same, I did not bother changing it, just made a notebook from it.

In [1]:
import os
import datetime
import calendar

import numpy as np
import pandas as pd

In [2]:
random_seed = 42 # answer to all things in the universe xD

In [3]:
PRODUCTS = {
    "iPhone": [700, 12],
    "Google Phone": [600, 9],
    "Samsung Phone": [650, 12],
    "20in Monitor": [109.99, 8],
    "34in Ultrawide Monitor": [379.99, 10],
    "27in 4K Gaming Monitor": [389.99, 8],
    "27in FHD Monitor": [149.99, 14],
    "Flatscreen TV": [300, 8],
    "Macbook Pro": [1699, 9],
    "Macbook Air": [1200, 6],
    "Dell Laptop": [999.99, 7],
    "Lenovo Laptop": [899.99, 4],
    "AA Batteries (4-pack)": [3.84, 40],
    "AAA Batteries (4-pack)": [2.99, 40],
    "USB-C Charging Cable": [11.95, 30],
    "Lightning Charging Cable": [14.95, 30],
    "Wired Headphones": [11.99, 30],
    "Bose SoundSport Headphones": [99.99, 25],
    "Apple Airpods Headphones": [150, 25],
    "Gaming Mouse": [100, 15],
    "Mechanical Keyboard": [250, 10],
    "Normal Keyboard": [99.99, 15],
    "Cooling Pad": [29.99, 10],
    "LG Washing Machine": [600.00, 1],
    "LG Dryer": [600.00, 1],
}

OUTPUT_COLUMNS = [
    "Order ID",
    "Product",
    "Quantity Ordered",
    "Price Each",
    "Order Date",
    "Purchase Address",
]

STREET_NAMES = [
    "Main",
    "2nd",
    "1st",
    "4th",
    "5th",
    "Park",
    "6th",
    "7th",
    "Maple",
    "Pine",
    "Washington",
    "8th",
    "Cedar",
    "Elm",
    "Walnut",
    "9th",
    "10th",
    "Lake",
    "Sunset",
    "Lincoln",
    "Jackson",
    "Church",
    "River",
    "11th",
    "Willow",
    "Jefferson",
    "Center",
    "12th",
    "North",
    "Lakeview",
    "Ridge",
    "Hickory",
    "Adams",
    "Cherry",
    "Highland",
    "Johnson",
    "South",
    "Dogwood",
    "West",
    "Chestnut",
    "13th",
    "Spruce",
    "14th",
    "Wilson",
    "Meadow",
    "Forest",
    "Hill",
    "Madison",
]

CITIES = [
    "San Francisco",
    "Boston",
    "New York City",
    "Austin",
    "Dallas",
    "Atlanta",
    "Portland",
    "Portland",
    "Los Angeles",
    "Seattle",
]

CITY_WEIGHTS = np.array([9, 4, 5, 2, 3, 3, 2, 0.5, 6, 3], dtype=float)

STATES = ["CA", "MA", "NY", "TX", "TX", "GA", "OR", "ME", "CA", "WA"]

ZIP_CODES = ["94016", "02215", "10001", "73301", "75001", "30301", "97035", "04101", "90001", "98101"]

In [4]:
def generate_random_day(month, rng, year=2019):
    day_range = calendar.monthrange(year, month)[1]
    return int(rng.integers(1, day_range + 1))


def generate_random_time(month, rng, year=2019):
    day = generate_random_day(month, rng, year)

    if rng.random() < 0.5:
        base_date = datetime.datetime(year, month, day, 12, 0)
    else:
        base_date = datetime.datetime(year, month, day, 20, 0)

    time_offset_minutes = rng.normal(loc=0.0, scale=180.0)
    final_date = base_date + datetime.timedelta(minutes=float(time_offset_minutes))

    return final_date.strftime("%m/%d/%y %H:%M")


def generate_random_address(rng):
    street_name = rng.choice(STREET_NAMES)
    city_position = rng.choice(len(CITIES), p=CITY_WEIGHTS / CITY_WEIGHTS.sum())

    street_number = int(rng.integers(1, 1000))
    city = CITIES[city_position]
    state = STATES[city_position]
    zip_code = ZIP_CODES[city_position]

    return f"{street_number} {street_name} St, {city}, {state} {zip_code}"


In [5]:
def sample_month_order_count(month, rng):
    if month == 1:
        return int(rng.normal(loc=10000, scale=3000))

    if month == 2:
        return int(rng.normal(loc=15000, scale=2000))

    if month in range(3, 10):
        return int(rng.normal(loc=12000, scale=2000))

    if month == 10:
        return int(rng.normal(loc=15000, scale=3000))

    if month == 11:
        return int(rng.normal(loc=20000, scale=3000))

    return int(rng.normal(loc=26000, scale=2000))


def make_order_row(order_number, product_name, order_date, address, rng):
    product_price = PRODUCTS[product_name][0]
    quantity_ordered = int(rng.geometric(p=1.0 - (1.0 / product_price)))

    return [
        order_number,
        product_name,
        quantity_ordered,
        product_price,
        order_date,
        address,
    ]


def add_order_row(rows, order_number, product_name, order_date, address, rng):
    rows.append(make_order_row(order_number, product_name, order_date, address, rng))

In [6]:
def add_related_products(rows, order_number, product_name, order_date, address, rng):
    if product_name == "iPhone":
        if rng.random() < 0.15:
            add_order_row(rows, order_number, "Lightning Charging Cable", order_date, address, rng)
        if rng.random() < 0.07:
            add_order_row(rows, order_number, "Apple Airpods Headphones", order_date, address, rng)

    elif product_name in ["Google Phone", "Samsung Phone"]:
        if rng.random() < 0.18:
            add_order_row(rows, order_number, "USB-C Charging Cable", order_date, address, rng)
        if rng.random() < 0.05:
            add_order_row(rows, order_number, "Bose SoundSport Headphones", order_date, address, rng)
        if rng.random() < 0.07:
            add_order_row(rows, order_number, "Wired Headphones", order_date, address, rng)

    elif product_name in ["Macbook Pro", "Macbook Air"]:
        if rng.random() < 0.16:
            add_order_row(rows, order_number, "Normal Keyboard", order_date, address, rng)
        if rng.random() < 0.04:
            add_order_row(rows, order_number, "Apple Airpods Headphones", order_date, address, rng)

    elif product_name in ["Dell Laptop", "Lenovo Laptop"]:
        if rng.random() < 0.13:
            add_order_row(rows, order_number, "Cooling Pad", order_date, address, rng)
        if rng.random() < 0.05:
            add_order_row(rows, order_number, "Gaming Mouse", order_date, address, rng)
        if rng.random() < 0.08:
            add_order_row(rows, order_number, "Wired Headphones", order_date, address, rng)

    elif product_name in ["27in 4K Gaming Monitor", "34in Ultrawide Monitor"]:
        if rng.random() < 0.12:
            add_order_row(rows, order_number, "Mechanical Keyboard", order_date, address, rng)
        if rng.random() < 0.04:
            add_order_row(rows, order_number, "Wired Headphones", order_date, address, rng)
        if rng.random() < 0.05:
            add_order_row(rows, order_number, "Bose SoundSport Headphones", order_date, address, rng)
        if rng.random() < 0.08:
            add_order_row(rows, order_number, "Gaming Mouse", order_date, address, rng)


In [7]:
def generate_month_sales(month, starting_order_number, year, rng):
    order_count = max(0, sample_month_order_count(month, rng))
    product_names = list(PRODUCTS.keys())
    product_weights = np.array([PRODUCTS[product][1] for product in product_names], dtype=float)

    rows = []
    order_number = starting_order_number

    for _ in range(order_count):
        address = generate_random_address(rng)
        order_date = generate_random_time(month, rng, year)

        product_name = rng.choice(product_names, p=product_weights / product_weights.sum())

        add_order_row(rows, order_number, product_name, order_date, address, rng)
        add_related_products(rows, order_number, product_name, order_date, address, rng)

        if rng.random() <= 0.02:
            extra_product_name = rng.choice(product_names, p=product_weights / product_weights.sum())
            add_order_row(rows, order_number, extra_product_name, order_date, address, rng)

        order_number += 1

    month_df = pd.DataFrame(rows, columns=OUTPUT_COLUMNS)

    return month_df, order_number


In [8]:
def simulate_sales_year(config, seed=random_seed):
    rng = np.random.default_rng(seed)

    monthly_sales = {}
    all_sales = []

    order_number = config["starting_order_number"]
    year = config["year"]

    for month in range(1, 13):
        month_name = calendar.month_name[month]
        month_df, order_number = generate_month_sales(month, order_number, year, rng)

        monthly_sales[month_name] = month_df
        all_sales.append(month_df)

    sales_df = pd.concat(all_sales, ignore_index=True)

    return sales_df, monthly_sales


def export_sales_files(sales_df, monthly_sales, output_directory, year):
    os.makedirs(output_directory, exist_ok=True)
    # for month_name, month_df in monthly_sales.items():
    #     month_df.to_csv(f"{output_directory}/Sales_{month_name}_{year}.csv", index=False)

    sales_df.to_csv(f"{output_directory}/Sales_{year}_All_Months.csv", index=False)

In [9]:
sales_config = {
    "year": 2019,
    "starting_order_number": 145345,
    "output_directory": "../generated_data/tabular/ecommerce_sales_2019",
}

In [10]:
sales_df, monthly_sales = simulate_sales_year(sales_config, seed=random_seed)

export_sales_files(
    sales_df=sales_df,
    monthly_sales=monthly_sales,
    output_directory=sales_config["output_directory"],
    year=sales_config["year"],
)

sales_df.head()

,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address
0,145345,Bose SoundSport Headphones,1,99.99,01/03/19 08:05,"439 Hickory St, Los Angeles, CA 90001"
1,145346,Apple Airpods Headphones,1,150.00,01/12/19 20:11,"500 Cherry St, New York City, NY 10001"
2,145347,AA Batteries (4-pack),3,3.84,01/27/19 19:26,"555 5th St, San Francisco, CA 94016"
3,145348,27in 4K Gaming Monitor,1,389.99,01/07/19 13:05,"760 Chestnut St, Los Angeles, CA 90001"
4,145349,AAA Batteries (4-pack),2,2.99,01/22/19 13:57,"190 West St, San Francisco, CA 94016"


In [11]:
sales_df.shape

(178499, 6)

In [12]:
sales_df["Product"].value_counts().head(10)

Product
AAA Batteries (4-pack)        18168
AA Batteries (4-pack)         17850
USB-C Charging Cable          14976
Wired Headphones              14758
Lightning Charging Cable      14258
Bose SoundSport Headphones    12073
Apple Airpods Headphones      11896
Normal Keyboard                7685
Gaming Mouse                   7573
27in FHD Monitor               6253
Name: count, dtype: int64

In [13]:
monthly_row_counts = pd.Series({
    month_name: len(month_df)
    for month_name, month_df in monthly_sales.items()
})

monthly_row_counts

January      11691
February     14275
March        15030
April         7290
May          14705
June         17802
July         13343
August       10897
September    11731
October      15516
November     18571
December     27648
dtype: int64

In [14]:
sales_df.groupby("Product")["Quantity Ordered"].agg(["count", "mean", "max"]).sort_values("count", ascending=False).head(10)

,count,mean,max
Product,,,
AAA Batteries (4-pack),18168,1.508917,10
AA Batteries (4-pack),17850,1.347899,7
USB-C Charging Cable,14976,1.094551,5
Wired Headphones,14758,1.088156,5
Lightning Charging Cable,14258,1.072380,4
Bose SoundSport Headphones,12073,1.010768,2
Apple Airpods Headphones,11896,1.006809,2
Normal Keyboard,7685,1.010020,2
Gaming Mouse,7573,1.009640,3


In [15]:
sales_df["Purchase Address"].str.extract(r", ([A-Za-z ]+), [A-Z]{2}")[0].value_counts()

0
San Francisco    42725
Los Angeles      28736
New York City    23828
Boston           19023
Dallas           14293
Seattle          14150
Atlanta          14135
Portland         11854
Austin            9755
Name: count, dtype: int64